# WS-9 v4 — HUMANIZAÇÃO dos parâmetros (âncoras Groveman 2019) + células granulares

**Objetivo:** substituir o relógio murino (Params.csv Igel) por âncoras de organoide HUMANO infectado com sCJD (Groveman 2019, PMC6567389).

**Âncoras humanas (do texto — valores citados):**
- Inóculo clareado 25-28 dpi · produção de novo a partir de 35 dpi
- Título final (169 dpi): MV2 = 2,13e5 · MV1 = 1,69e3 SD50/mg → MV2 ≈ 126× MV1
- WB PrP-res positivo apenas no MV2 · janela de crescimento 35→169 dpi

**Método honesto:** θ é invariante sob reescala uniforme do tempo → a humanização muda o CALENDÁRIO (dias reais por unidade sim) e a AMPLITUDE (MV1 vs MV2); θ* reavaliado em ambas (banda, não ponto).

**Granularidade:** células curtas; as longas imprimem progresso a cada 20% com elapsed/ETA. VM caiu → retome da célula parada.

In [ ]:
#@title C0 — motor v4 (PROGRESSO interno + duplicação) ~5s {display-mode:"form"}
import numpy as np, json, math, os, time
os.makedirs('/content/out', exist_ok=True); T0=time.time()

def simulate(div=96, s=10, t_lim=5.0, dt=5e-4, nrec=80, kcap=0.0, ell_mm=3.6,
             Kt=(10.0,5.0), Kr=(50.0,10.0), Kc=(10.0,50.0), D0=1000.0, L=1.0,
             uprd=5.0, uprt=10.0, uprr=6.0, tpr=10.0, C50=50.0, seed_mass=130.0,
             tag='', progress=False):
    t_start=time.time()
    px_per_mm=div/4.0
    K_templ,K_auto,K_nucl,K_frag,K_decond,K_cond=Kt[1],Kt[0],Kr[0],Kr[1],Kc[0],Kc[1]
    K1=4*D0/(np.arange(1,s+2)*L**2)
    X,Y=np.meshgrid(np.arange(1,div+1),np.arange(1,div+1))
    m_lin=round(uprr+(div-2*uprr)/2); step=max(1,round((div-2*uprr-1)/2))
    neur=[(m_lin+step*i,m_lin+step*j) for i in(-1,0,1) for j in(-1,0,1)]
    def disk(cx,cy,r): return ((X-cx)**2+(Y-cy)**2)<r**2
    tpl=[disk(*p,tpr) for p in neur]; upz=[disk(*p,uprr) for p in neur]
    c0=(div//2,div//2)
    P=np.zeros((div,div,s+1),dtype=float)
    sm=disk(c0[0],c0[1],3.0); P[sm,s]=seed_mass/sm.sum()
    rr=np.hypot(X-c0[0],Y-c0[1])/px_per_mm
    cV=np.exp(-rr/ell_mm) if kcap>0 else np.zeros((div,div))
    upr_t=np.zeros(9); upr_on=np.zeros(9,bool); tp=np.ones((div,div))
    steps=int(t_lim/dt); rec_every=max(1,steps//nrec)
    marks={int(steps*f):f for f in (0.2,0.4,0.6,0.8)}
    T=[];TOT=[];R=[];U=[]
    for st in range(steps):
        # EARLY-STOP físico: extinção (carga < semente/1e6) interrompe e economiza minutos
        if st%400==0 and st>0 and kcap>0 and P.sum()<seed_mass*1e-6:
            print(f'  [{tag}] EXTINÇÃO detectada no passo {st}/{steps} — encerrando cedo', flush=True)
            for k2 in range(st,steps):
                if k2%rec_every==0:
                    T.append(k2*dt); TOT.append(float(P.sum()))
                    ys,xs=np.nonzero(P.sum(axis=2)>1e-9)
                    R.append(float(np.max(np.hypot(xs-c0[0],ys-c0[1]))/px_per_mm) if len(xs) else 0.)
                    U.append(float(upr_on.mean()))
            break
        if st%20==0:
            for n,(um,tm) in enumerate(zip(upz,tpl)):
                if P[um].sum()>uprd:
                    upr_t[n]+=dt*20
                    if upr_t[n]>=uprt: upr_on[n]=True
            tp.fill(1.0)
            for n,(um,tm) in enumerate(zip(upz,tpl)):
                if upr_on[n]: tp[tm]=0.0
        eff=tp
        C=P[:,:,s]; dP=np.zeros_like(P)
        freeS=(1.0/(1.0+kcap*cV))**2 if kcap>0 else np.ones((div,div))
        dP[:,:,s]+=dt*K_auto*eff*C*(C/(C+C50))*freeS
        for a in range(s-1):
            g=dt*K_templ*eff*P[:,:,a]*freeS
            dP[:,:,a]-=g; dP[:,:,a+1]+=g
        nuc=dt*K_nucl*C*freeS
        dP[:,:,0]+=nuc; dP[:,:,s]-=nuc
        frs=dt*K_frag*C[:,:,None]*P[:,:,1:s]
        dP[:,:,1:s]-=frs; dP[:,:,0:s-1]+=frs; dP[:,:,s]+=frs.sum(axis=2)
        dcs=dt*K_decond*P[:,:,1:s]
        dP[:,:,1:s]-=dcs; dP[:,:,0:s-1]+=dcs; dP[:,:,0]+=dcs.sum(axis=2)
        for a in range(s):
            for b in range(max(1,1-a),s-a):
                cr=dt*K_cond*P[:,:,a]*P[:,:,b]/(div*div)*10
                dP[:,:,a]-=cr; dP[:,:,b]-=cr; dP[:,:,a+b]+=2*cr
        lap=(np.roll(P,1,0)+np.roll(P,-1,0)+np.roll(P,1,1)+np.roll(P,-1,1)-4*P)
        dP+=dt*K1[None,None,:]/(div*div)*lap
        P=np.clip(P+dP,0,1e6)
        if st%rec_every==0:
            T.append(st*dt); TOT.append(float(P.sum()))
            ys,xs=np.nonzero(P.sum(axis=2)>1e-9)
            R.append(float(np.max(np.hypot(xs-c0[0],ys-c0[1]))/px_per_mm) if len(xs) else 0.)
            U.append(float(upr_on.mean()))
        if progress and st in marks:
            el=time.time()-t_start
            print(f'  [{tag}] {int(marks[st]*100)}%  elapsed {el:.0f}s  ETA {el/st*(steps-st):.0f}s', flush=True)
    tot=np.array(TOT); tarr=np.array(T)
    lt=np.log(np.clip(tot,1e-9,None)); rng=lt.max()-lt.min()
    m=(lt>lt.min()+0.1*rng)&(lt<lt.max()-0.1*rng)&(tarr>0)
    t_double=None; r2=None
    if m.sum()>5:
        A=np.polyfit(tarr[m],lt[m],1)
        if A[0]>0: t_double=math.log(2)/A[0]; r2=float(np.corrcoef(tarr[m],lt[m])[0,1]**2)
    return dict(t=T,total=TOT,r=R,upr=U,final_R_mm=R[-1] if R else 0.,
                total0=TOT[0],totalf=TOT[-1] if TOT else 0,
                t_double_sim=t_double,fit_r2=r2,wall_s=time.time()-t_start)
print('motor v4 OK — com progresso interno e t_double')

In [ ]:
#@title C1 — âncoras humanas (Groveman 2019) + duplicação-alvo — instantâneo {display-mode:"form"}
ANCH={'clear_dpi':25.5,'first_denovo_dpi':35,'final_dpi':169,
      'titer_MV2':2.13e5,'titer_MV1':1.69e3,'WB_MV2_only':True}
import math as _m
growth_window=ANCH['final_dpi']-ANCH['first_denovo_dpi']
doublings=_m.log2(ANCH['titer_MV2']/100.0)
t_double_h=growth_window/doublings
print(f"eclipse→{ANCH['first_denovo_dpi']} dpi; crescimento {ANCH['first_denovo_dpi']}→{ANCH['final_dpi']} dpi")
print(f"título MV2={ANCH['titer_MV2']:.2e} vs MV1={ANCH['titer_MV1']:.2e} (razão {ANCH['titer_MV2']/ANCH['titer_MV1']:.0f}x)")
print(f"=> ~{doublings:.1f} dobras na janela → t_dupl HUMANO ≈ {t_double_h:.1f} dias (piso detecção 1e2 — suposição documentada)")

In [ ]:
#@title C2 — self-test mini-malha com progresso (~10s) {display-mode:"form"}
r=simulate(div=32,t_lim=1.0,nrec=20,tag='SELFTEST',progress=True)
print(f"mini OK | total {r['total0']:.0f}→{r['totalf']:.1e} | t_double={r['t_double_sim'] and round(r['t_double_sim'],3)}")
assert r['totalf']>r['total0'],'motor quebrado'


In [ ]:
#@title C3 — BASELINE MV2-like (progresso) ~1-2 min {display-mode:"form"}
print('baseline MV2-like (seed_mass=130)...')
BASE=simulate(kcap=0.0, seed_mass=130.0, tag='BASE-MV2', progress=True)
T1=BASE['totalf']>BASE['total0']*1.5
td=BASE['t_double_sim'] and round(BASE['t_double_sim'],3); r2=BASE['fit_r2'] and round(BASE['fit_r2'],3)
print(f"T1={T1} | total {BASE['total0']:.0f}→{BASE['totalf']:.2e} | R={BASE['final_R_mm']:.2f}mm | t_double_sim={td} r2={r2}")
assert T1,'ABORTAR: baseline não replica'


In [ ]:
#@title C4 — CALIBRAÇÃO DO RELÓGIO — instantâneo {display-mode:"form"}
assert BASE['t_double_sim'] and BASE['t_double_sim']>0,'regressão de duplicação falhou'
days_per_simunit=t_double_h/BASE['t_double_sim']
sim_days=5.0*days_per_simunit
print(f"t_double_sim = {BASE['t_double_sim']:.3f} u  |  t_double_humano = {t_double_h:.1f} d")
print(f"=> 1 unidade sim = {days_per_simunit:.2f} dias reais")
print(f"=> sim completa ≈ {sim_days:.0f} dias (~{sim_days/30.4:.1f} meses) vs janela real 169 dpi ≈ 5,6 meses")
print(f"=> G0 deve ler contenção/halo em ~{min(sim_days,169):.0f} dias pós-seeding")

In [ ]:
#@title C5 — T2 guard-rail: k=32 contém? ~2-4 min (com early-stop de extinção) {display-mode:"form"}
R32=simulate(kcap=32.0, seed_mass=130.0, tag='k32', progress=True)
T2=R32['final_R_mm']<0.9*BASE['final_R_mm']
print(f"R(k=32)={R32['final_R_mm']:.2f} vs base {BASE['final_R_mm']:.2f} | T2={T2}")
assert T2,'capping ineficaz — abortar'


In [ ]:
#@title C6 — k=2 (limiar inferior v2) ~1-2 min {display-mode:"form"}
R2=simulate(kcap=2.0, seed_mass=130.0, tag='k2', progress=True)
print(f"k=2: R={R2['final_R_mm']:.2f}mm ratio={min(R2['totalf']/R2['total0'],1e9):.1e}")

In [ ]:
#@title C7 — k=3 (refinamento do limiar) ~1-2 min {display-mode:"form"}
R3=simulate(kcap=3.0, seed_mass=130.0, tag='k3', progress=True)
print(f"k=3: R={R3['final_R_mm']:.2f}mm ratio={min(R3['totalf']/R3['total0'],1e9):.1e}")

In [ ]:
#@title C8 — k=4 (limiar superior v2) ~1-2 min {display-mode:"form"}
R4=simulate(kcap=4.0, seed_mass=130.0, tag='k4', progress=True)
print(f"k=4: R={R4['final_R_mm']:.2f}mm ratio={min(R4['totalf']/R4['total0'],1e9):.1e}")

In [ ]:
#@title C9 — k=8 ~1-2 min (extinção → early-stop) {display-mode:"form"}
R8=simulate(kcap=8.0, seed_mass=130.0, tag='k8', progress=True)
print(f"k=8: R={R8['final_R_mm']:.2f}mm ratio={min(R8['totalf']/R8['total0'],1e9):.1e}")

In [ ]:
#@title C10 — MV1-like (semente 126x menor) ~2x1-2 min {display-mode:"form"}
seed_mv1=130.0/126.0
B1=simulate(kcap=0.0, seed_mass=seed_mv1, tag='BASE-MV1', progress=True)
R4M1=simulate(kcap=4.0, seed_mass=seed_mv1, tag='k4-MV1', progress=True)
print(f"MV1-like: T1={B1['totalf']>B1['total0']*1.5} | base R={B1['final_R_mm']:.2f} | k=4 R={R4M1['final_R_mm']:.2f}")
print("(esperado: cresce menos — coerente com WB-negativo — e o mesmo k contém mais fácil)")

In [ ]:
#@title C11 — MERGE FINAL: theta*_humano + calendário + downloads ~15s {display-mode:"form"}
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,3,figsize=(16,4.5))
runs={'MV2':{0.0:BASE,2.0:R2,3.0:R3,4.0:R4,8.0:R8,32.0:R32},'MV1':{0.0:B1,4.0:R4M1}}
for lbl,rs in runs.items():
    ax[0].plot(rs[0.0]['t'],rs[0.0]['r'],lw=2.5,label=lbl+' baseline')
    ks=sorted(k for k in rs if k>0)
    for k in ks: ax[0].plot(rs[k]['t'],rs[k]['r'],lw=1.3,alpha=0.8,label=lbl+f' k={k:g}')
    th=[1/(1+k) for k in ks]; RR=[rs[k]['final_R_mm'] for k in ks]
    ax[1].semilogx(th,RR,'o-',label=lbl)
ax[0].set_xlabel('t sim'); ax[0].set_ylabel('frente mm'); ax[0].legend(fontsize=6); ax[0].set_title('frentes MV2/MV1')
ax[1].axhline(BASE['final_R_mm'],ls='--',c='gray'); ax[1].set_xlabel('theta'); ax[1].set_ylabel('R final mm')
ax[1].legend(); ax[1].set_title('resposta humanizada')
ax[2].semilogy(BASE['t'],np.clip(BASE['total'],1e-2,None),'k-',lw=2)
ax[2].set_xlabel(f't sim (={days_per_simunit:.1f} d/unid)'); ax[2].set_ylabel('carga (log)')
ax[2].set_title(f't_dupl humano ≈ {t_double_h:.0f} d')
plt.tight_layout(); plt.savefig('/content/out/ws_9_v4_human.png',dpi=140); plt.show()
Rb=BASE['final_R_mm']
sw=[dict(kappa=k,theta=round(1/(1+k),3),R_mm=round(runs['MV2'][k]['final_R_mm'],2)) for k in (2.0,3.0,4.0,8.0,32.0)]
star=[x for x in sw if x['R_mm']<0.5*Rb]
theta_star=star[0]['theta'] if star else None
out={'motor':'v4 humano (clock Groveman 2019)','days_per_simunit':round(days_per_simunit,2),
     't_double_human_days':round(t_double_h,1),'anchors':ANCH,'sweep_MV2':sw,'theta_star':theta_star,
     'MV1':{'kappa4_R_mm':round(R4M1['final_R_mm'],2),'baseline_R_mm':round(B1['final_R_mm'],2)},
     'wall_total_s':round(time.time()-T0,1)}
json.dump(out,open('/content/out/ws_9_v4_human.json','w'),indent=1)
print('theta* humano (MV2, R<50%base):',theta_star)
print('salvo /content/out/ws_9_v4_human.{json,png}')
from google.colab import files
files.download('/content/out/ws_9_v4_human.json'); files.download('/content/out/ws_9_v4_human.png')